In [2]:
import pandas as pd
import plotly.express as px

In [3]:
df = pd.read_csv("data/team_week_points.csv")
df = df[df.manager.notnull()]
len(df)

1270

In [ ]:
#3 — Points For vs. Points Against (Luck Scatter)

# In standings_line.py or a new page
import plotly.graph_objects as go

s = pd.read_csv("data/standings_data.csv")
s = s[s.manager.notnull()]

# Filter to final week of selected season
final = s[s['season'] == selected_year]
final = final[final['week'] == final['week'].max()]

ref = max(final['points_for'].max(), final['points_against'].max())

fig = px.scatter(
    final, x='points_against', y='points_for',
    text='manager', color='standings_rank',
    color_continuous_scale='RdYlGn_r',
    labels={'points_for': 'Points Scored (PS)', 'points_against': 'Points Allowed (PA)'},
    title=f"Luck Chart — {selected_year}"
)
# Diagonal = perfectly even PS/PA
fig.add_shape(type='line', x0=0, y0=0, x1=ref, y1=ref,
              line=dict(color='gray', dash='dash'))
fig.update_traces(textposition='top center', marker=dict(size=12))
fig.update_coloraxes(showscale=False)
st.plotly_chart(fig, use_container_width=True)

In [ ]:
#5 — Radar / Spider Chart (Manager Stat Fingerprint)
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler  # pip install scikit-learn

h = pd.read_csv("data/hitting_stats.csv")
p = pd.read_csv("data/pitching_stats.csv")

HIT_COLS = ['pts', 'R',  'RBI', 'BB', 'HR',   'TB',      'SB']
PIT_COLS = ['pts', 'IP', 'K',   'QS', 'SV+H', 'K_per_IP']  # keeping ERA/WHIP out since lower=better complicates scaling


selected_year = st.selectbox("Select year", sorted(h['season'].unique(), reverse=True))
managers = st.multiselect("Select managers", h['manager'].unique())

h_agg = h[h['season'] == selected_year].groupby('manager')[HIT_COLS].sum()
p_agg = p[p['season'] == selected_year].groupby('manager')[PIT_COLS].sum()

scaler = MinMaxScaler()
hit_scaled = pd.DataFrame(scaler.fit_transform(h_agg), index=h_agg.index, columns=h_agg.columns)
pit_scaled = pd.DataFrame(scaler.fit_transform(p_agg), index=p_agg.index, columns=p_agg.columns)

fig = go.Figure()
for manager in managers:
    ## hit compare
    categories = list(hit_scaled.columns)
    vals = hit_scaled.loc[manager].tolist()
    vals += [vals[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=categories + [categories[0]],
        fill='toself', name=manager
    ))

fig.update_layout(polar=dict(radialaxis=dict(visible=False, range=[0, 1])),
                  title=f"Manager Stat Profile - {selected_year}")
fig.show()


fig2 = go.Figure()
for manager in managers:
    ## pitch compare
    categories = list(pit_scaled.columns)
    vals = pit_scaled.loc[manager].tolist()
    vals += [vals[0]]
    fig2.add_trace(go.Scatterpolar(
        r=vals, theta=categories + [categories[0]],
        fill='toself', name=manager
    ))

fig2.update_layout(polar=dict(radialaxis=dict(visible=False, range=[0, 1])),
                  title=f"Manager Stat Profile - {selected_year}")

col1, col2 = st.columns(2)
with col1:
    st.plotly_chart(fig, use_container_width=True)

with col2:
    st.plotly_chart(fig2, use_container_width=True)

In [4]:
import numpy as np

In [5]:
from utils import style_pts

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

SEASON = 2025
WEEK = 2
tmp = hit.copy()
highlight_mask = (tmp['season'] == SEASON) & (tmp['week'] == WEEK)
tmp['_dummy_x'] = 0

for col in ['R', 'RBI', 'R', 'SB', 'BB', 'HBP', '1B', '2B', '3B', 'HR', 'TB']:
    fig = px.strip(tmp,
                x='_dummy_x',
                y=col,
                color_discrete_sequence=['gray'],
                hover_data={'season': True, 'week': True, col: True, '_dummy_x': False},  # Only show what you want
                )
    fig.data[0].hovertemplate = 'Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>' + col + ': %{y:.3f}<extra></extra>'
    fig.data[0].customdata = tmp[['season', 'week']].values

    # Overlay highlights in red
    fig.add_scatter(
        x=tmp.loc[highlight_mask, '_dummy_x'],
        y=tmp.loc[highlight_mask, col],
        mode='markers',
        marker=dict(color='red', size=12, symbol='star'),
        hovertemplate='Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>' + col + ': %{y:.3f}<extra></extra>',
        customdata=tmp.loc[highlight_mask, ['season', 'week']].values,

    )
    fig.update_xaxes(range=[-0.2, 0.2], showticklabels=False, title_text='')
    fig.update_layout(showlegend=False)
    fig.show()